# **Avance_4: Arquitectura en la Nube**

Se analizará el código de las tres funciones Lambda para entender la arquitectura del proyecto.
También se analizará el código que permite construir la arquitectura en AWS.

## **1. Funciones Lambda**

En total son 3 funciones AWS Lambda desarrolladas en Python para la arquitectura en la nube del sistema FleetLogix.

In [ ]:
# Recibe las confirmaciones de entrega enviadas desde la App Móvil de los conductores a través de AWS API Gateway. 
# Valida la información recibida y actualiza el estado de la entrega en la base de datos NoSQL Amazon DynamoDB.

import json                         # Permite manipular, serializar y deserializar datos en formato JSON (necesario para respuestas de API y eventos)
import boto3                        # AWS para Python; permite interactuar con servicios como DynamoDB, Kinesis, S3 y SNS
import os                           # Permite acceder a variables de entorno del sistema (como nombres de tablas o ARNs de SNS)
from datetime import datetime       # Permite generar y manipular marcas de tiempo (timestamps) para registrar fechas de entregas o eventos

# Clientes AWS - Conexión con Cuenta AWS
dynamodb = boto3.resource('dynamodb')
TABLE_NAME = os.environ.get('DELIVERIES_TABLE', 'deliveries_status')
table = dynamodb.Table(TABLE_NAME)

### **1.1. Funcion uno: Verificar Entrega**

Permite verificar si una entrega se completó comparando con operación en DynamoDB

1. Lambda 1 - Verificar Entrega:
   - Trigger: API Gateway POST /deliveries/verify
   - Ejecuta cada vez que app móvil marca entrega
   

In [ ]:
# =====================================================
# LAMBDA 1: Verificar si una entrega se completó 
# =====================================================

def lambda_verificar_entrega(event, context):
    """
    Verifica si una entrega se completó comparando con DynamoDB
    """
    
    # Obtener datos del evento
    delivery_id = event.get('delivery_id')
    tracking_number = event.get('tracking_number')
    
    if not delivery_id:
        return {
            'statusCode': 400,
            'body': json.dumps({'error': 'delivery_id es requerido'})
        }
    
    # Conectar a tabla DynamoDB
    table = dynamodb.Table('deliveries_status')
    
    try:
        # Buscar entrega
        response = table.get_item(
            Key={'delivery_id': delivery_id}
        )
        
        if 'Item' in response:
            item = response['Item']
            is_completed = item.get('status') == 'delivered'
            
            return {
                'statusCode': 200,
                'body': json.dumps({
                    'delivery_id': delivery_id,
                    'tracking_number': item.get('tracking_number'),
                    'is_completed': is_completed,
                    'status': item.get('status'),
                    'delivered_datetime': str(item.get('delivered_datetime', ''))
                })
            }
        else:
            return {
                'statusCode': 404,
                'body': json.dumps({
                    'error': 'Entrega no encontrada',
                    'delivery_id': delivery_id
                })
            }
            
    except Exception as e:
        return {
            'statusCode': 500,
            'body': json.dumps({
                'error': str(e)
            })
        }

### **1.2. Funcion dos: Calcular ETA**

Calcula el tiempo estimado de llegada (ETA - Estimated Time of Arrival) de un vehículo a su siguiente punto de entrega basándose en la distancia restante y la velocidad promedio en tiempo real.

2. Lambda 2 - Calcular ETA:
   - Trigger: EventBridge cada 5 minutos
   - Procesa todos los vehículos en ruta
   

In [ ]:
# =====================================================
# LAMBDA 2: Calcular tiempo estimado de llegada (ETA)
# =====================================================
def lambda_calcular_eta(event, context):
    """
    Calcula ETA basado en ubicación actual y destino
    """
    
    # Obtener datos del evento
    vehicle_id = event.get('vehicle_id')
    current_location = event.get('current_location')  # {lat, lon}
    destination = event.get('destination')  # {lat, lon}
    current_speed_kmh = event.get('current_speed_kmh', 60)
    
    if not all([vehicle_id, current_location, destination]):
        return {
            'statusCode': 400,
            'body': json.dumps({'error': 'Faltan parámetros requeridos'})
        }
    
    try:
        # Calcular distancia simple (Haversine simplificado)
        lat_diff = abs(destination['lat'] - current_location['lat'])
        lon_diff = abs(destination['lon'] - current_location['lon'])
        
        # Aproximación simple: 111 km por grado
        distance_km = ((lat_diff ** 2 + lon_diff ** 2) ** 0.5) * 111
        
        # Calcular tiempo
        if current_speed_kmh > 0:
            hours = distance_km / current_speed_kmh
            eta = datetime.now() + timedelta(hours=hours)
        else:
            eta = None
        
        # Guardar en DynamoDB
        table = dynamodb.Table('vehicle_tracking')
        table.put_item(
            Item={
                'vehicle_id': vehicle_id,
                'timestamp': datetime.now().isoformat(),
                'current_location': current_location,
                'destination': destination,
                'distance_remaining_km': Decimal(str(round(distance_km, 2))),
                'eta': eta.isoformat() if eta else None,
                'current_speed_kmh': Decimal(str(current_speed_kmh))
            }
        )
        
        return {
            'statusCode': 200,
            'body': json.dumps({
                'vehicle_id': vehicle_id,
                'distance_remaining_km': round(distance_km, 2),
                'eta': eta.isoformat() if eta else 'No disponible',
                'estimated_minutes': round(hours * 60) if eta else None
            })
        }
        
    except Exception as e:
        return {
            'statusCode': 500,
            'body': json.dumps({
                'error': str(e)
            })
        }

### **1.3. Funcion tres: Alerta de Desvío**

Recibe las coordenadas en tiempo real enviadas por Amazon Kinesis Data Streams. Compara la posición actual del vehículo contra el waypoint planeado. Si la distancia supera el umbral máximo tolerado (ej. 2 km), publica una alerta inmediata en Amazon SNS para notificar al equipo de monitoreo.

3. Lambda 3 - Alertas Desvío:
   - Trigger: Kinesis Stream de GPS
   - Ejecuta con cada actualización de ubicación

In [ ]:
# =====================================================
# LAMBDA 3: Enviar alerta si camión se desvía de ruta
# =====================================================
def lambda_alerta_desvio(event, context):
    """
    Detecta desvíos de ruta y envía alertas
    """
    
    # Obtener datos del evento
    vehicle_id = event.get('vehicle_id')
    current_location = event.get('current_location')  # {lat, lon}
    route_id = event.get('route_id')
    driver_id = event.get('driver_id')
    
    if not all([vehicle_id, current_location, route_id]):
        return {
            'statusCode': 400,
            'body': json.dumps({'error': 'Faltan parámetros requeridos'})
        }
    
    try:
        # Obtener ruta esperada de DynamoDB
        table = dynamodb.Table('routes_waypoints')
        response = table.get_item(
            Key={'route_id': route_id}
        )
        
        if 'Item' not in response:
            return {
                'statusCode': 404,
                'body': json.dumps({'error': 'Ruta no encontrada'})
            }
        
        waypoints = response['Item'].get('waypoints', [])
        
        # Calcular distancia mínima a la ruta
        min_distance = float('inf')
        for waypoint in waypoints:
            lat_diff = abs(waypoint['lat'] - current_location['lat'])
            lon_diff = abs(waypoint['lon'] - current_location['lon'])
            distance = ((lat_diff ** 2 + lon_diff ** 2) ** 0.5) * 111  # km
            min_distance = min(min_distance, distance)
        
        # Umbral de desvío: 5 km
        DEVIATION_THRESHOLD_KM = 5
        is_deviated = min_distance > DEVIATION_THRESHOLD_KM
        
        if is_deviated:
            # Enviar alerta SNS
            message = {
                'vehicle_id': vehicle_id,
                'driver_id': driver_id,
                'route_id': route_id,
                'deviation_km': round(min_distance, 2),
                'current_location': current_location,
                'timestamp': datetime.now().isoformat(),
                'alert_type': 'ROUTE_DEVIATION'
            }
            
            sns.publish(
                TopicArn='arn:aws:sns:us-east-1:123456789012:fleetlogix-alerts',
                Message=json.dumps(message),
                Subject='Alerta: Desvío de Ruta Detectado'
            )
            
            # Guardar alerta en DynamoDB
            alerts_table = dynamodb.Table('alerts_history')
            alerts_table.put_item(Item=message)
        
        return {
            'statusCode': 200,
            'body': json.dumps({
                'vehicle_id': vehicle_id,
                'is_deviated': is_deviated,
                'deviation_km': round(min_distance, 2),
                'alert_sent': is_deviated,
                'threshold_km': DEVIATION_THRESHOLD_KM
            })
        }
        
    except Exception as e:
        return {
            'statusCode': 500,
            'body': json.dumps({
                'error': str(e)
            })
        }

## **2. Infraestructura AWS**

Script para configurar servicios AWS básicos

### **2.1. Librerías y conexiones**

In [ ]:
# Librerias necesarias 

import boto3                    # Para conectarse a AWS
import json                     # Leer y escribir datos en JSON
import psycopg2                 # Para conectarse a PostgresSQL
from datetime import datetime   # Para manejar fechas y horas

# Configuración

AWS_REGION = 'us-east-1'            # Lugar donde se va a correr
RDS_INSTANCE_ID = 'fleetlogix-db'   # Nombre de base de datos
S3_BUCKET_NAME = 'fleetlogix-data'  # Nombre de bucket de S3

# Clientes AWS: creación de cinco conexiones a AWS 

rds = boto3.client('rds', region_name=AWS_REGION)           # Listar o reiniciar base de datos RDS
s3 = boto3.client('s3', region_name=AWS_REGION)             # Subir o bajar archivos de S3
dynamodb = boto3.client('dynamodb', region_name=AWS_REGION) # Leer o escribir en DynamoDB
lambda_client = boto3.client('lambda', region_name=AWS_REGION)  # Invocar otras lambdas
iam = boto3.client('iam')                                       # Revisar permisos IAM

### **2.2. Creaciación de instancias en RDS PostgresSQL**

Esta función permite crear la base de datos de FleetLogix en AWS.

In [ ]:
def crear_rds_postgresql():
    """Crear instancia RDS PostgreSQL"""
    print(" Creando RDS PostgreSQL...")
    
    try:
        response = rds.create_db_instance(
            DBInstanceIdentifier=RDS_INSTANCE_ID,           # fleetlogix-db
            DBInstanceClass='db.t3.micro',                  # Free tier
            Engine='postgres',                              # Crea un postgres
            EngineVersion='15.4',                           
            MasterUsername='fleetlogix_admin',              # Usuario
            MasterUserPassword='FleetLogix2024!',           # Cambiar en producción
            AllocatedStorage=20,                            # 20 GB
            StorageType='gp2',                              
            BackupRetentionPeriod=7,                        # Backups automáticos 7 días
            PreferredBackupWindow='03:00-04:00',            # Horario del Backups
            PreferredMaintenanceWindow='sun:04:00-sun:05:00', # Mantenimiento 
            PubliclyAccessible=True,                        # Cambiar a False para que no sea accesible a todos
            Tags=[
                {'Key': 'Project', 'Value': 'FleetLogix'},
                {'Key': 'Environment', 'Value': 'Development'}
            ]
        )
        print(f" RDS creado: {response['DBInstance']['DBInstanceIdentifier']}")
        # Si la base de datos ya existe no la duplica y si hay otro error lo imprime
    except rds.exceptions.DBInstanceAlreadyExistsFault:
        print("RDS ya existe")
    except Exception as e:
        print(f" Error creando RDS: {e}")

### **2.3. Creación de Bucket S3 para datos históricos**

Creación de Data Lake, base de almacenamiento de datos. Lo que lleve 90 días en raw-data se transfiere automáticamente a Glacier el cual es más económico como base de almacenamiento de datos

In [ ]:
def crear_s3_bucket():
    """Crear bucket S3 para datos históricos"""
    print("\nCreando S3 Bucket...")
    
    try:
        # Crear bucket
        s3.create_bucket(Bucket=S3_BUCKET_NAME)
        
        # Configurar estructura de carpetas
        folders = [
            'raw-data/',
            'processed-data/',
            'backups/',
            'logs/'
        ]
        
        for folder in folders:
            s3.put_object(
                Bucket=S3_BUCKET_NAME,
                Key=f"{folder}",
                Body=b''
            )
        
        # Configurar lifecycle para organizar por fecha
    
        lifecycle_config = {
            'Rules': [{
                'ID': 'archive-old-data',
                'Status': 'Enabled',
                'Transitions': [{
                    'Days': 90,
                    'StorageClass': 'GLACIER'
                }],
                'Prefix': 'raw-data/'
            }]
        }
        
        s3.put_bucket_lifecycle_configuration(
            Bucket=S3_BUCKET_NAME,
            LifecycleConfiguration=lifecycle_config
        )
        
        print(f" S3 Bucket creado: {S3_BUCKET_NAME}")
        
    except s3.exceptions.BucketAlreadyExists:
        print(" S3 Bucket ya existe")
    except Exception as e:
        print(f" Error creando S3: {e}")

### **2.4. Creación de tablas en DynamoDB**

Creación de tablas para consulta en tiempo real

In [ ]:
def crear_tablas_dynamodb():
    """Crear tablas DynamoDB para estado actual"""
    print("\n Creando tablas DynamoDB...")

# En esta tabla por cada entrega realizada se hace un registro y permite saber donde va el pedido  
    tablas = [
        {
            'TableName': 'deliveries_status',
            'KeySchema': [
                {'AttributeName': 'delivery_id', 'KeyType': 'HASH'}
            ],
            'AttributeDefinitions': [
                {'AttributeName': 'delivery_id', 'AttributeType': 'S'}
            ]
        },

# Permite guardar toda la ruta de un vehículo de transporte
        {
            'TableName': 'vehicle_tracking',
            'KeySchema': [
                {'AttributeName': 'vehicle_id', 'KeyType': 'HASH'},
                {'AttributeName': 'timestamp', 'KeyType': 'RANGE'}
            ],
            'AttributeDefinitions': [
                {'AttributeName': 'vehicle_id', 'AttributeType': 'S'},
                {'AttributeName': 'timestamp', 'AttributeType': 'S'}
            ]
        },

# Sirve para guardar la ruta planeada
        {
            'TableName': 'routes_waypoints',
            'KeySchema': [
                {'AttributeName': 'route_id', 'KeyType': 'HASH'}
            ],
            'AttributeDefinitions': [
                {'AttributeName': 'route_id', 'AttributeType': 'S'}
            ]
        },

# Genera alerta cuando un vehículo se sale de la ruta
        {
            'TableName': 'alerts_history',
            'KeySchema': [
                {'AttributeName': 'vehicle_id', 'KeyType': 'HASH'},
                {'AttributeName': 'timestamp', 'KeyType': 'RANGE'}
            ],
            'AttributeDefinitions': [
                {'AttributeName': 'vehicle_id', 'AttributeType': 'S'},
                {'AttributeName': 'timestamp', 'AttributeType': 'S'}
            ]
        }
    ]
    
    for tabla in tablas:
        try:
            response = dynamodb.create_table(
                TableName=tabla['TableName'],
                KeySchema=tabla['KeySchema'],
                AttributeDefinitions=tabla['AttributeDefinitions'],
                BillingMode='PAY_PER_REQUEST',  # On-demand Solo se paga lo que se use
                Tags=[
                    {'Key': 'Project', 'Value': 'FleetLogix'}
                ]
            )
            print(f" Tabla creada: {tabla['TableName']}")
            
        except dynamodb.exceptions.ResourceInUseException:
            print(f" Tabla ya existe: {tabla['TableName']}")
        except Exception as e:
            print(f" Error creando tabla {tabla['TableName']}: {e}")

### **2.5. Creación de Buckups automáticos para RDS**

Es una especie de snapshot que permite guardar datos por 7 días en caso de perdida de la base de datos. Este almacenamiento se aglomera eternamente consumiendo espacio, hasta que el usuario decida eliminarlo.

In [ ]:
def configurar_backups_automaticos():
    """Configurar backups automáticos para RDS"""
    print("\n⚙️ Configurando backups automáticos...")
    
    try:
        # Los backups ya están configurados en create_db_instance
        # Aquí podríamos agregar configuración adicional
        
        # Crear snapshot manual inicial
        snapshot_id = f"fleetlogix-initial-{datetime.now().strftime('%Y%m%d%H%M%S')}"
        
        rds.create_db_snapshot(
            DBSnapshotIdentifier=snapshot_id,
            DBInstanceIdentifier=RDS_INSTANCE_ID,
            Tags=[
                {'Key': 'Type', 'Value': 'Manual'},
                {'Key': 'Project', 'Value': 'FleetLogix'}
            ]
        )
        
        print(f" Snapshot inicial creado: {snapshot_id}")
        print(" Backups automáticos configurados (retención: 7 días)")
        
    except Exception as e:
        print(f" Error configurando backups: {e}")

### **2.5. Migración de PostgresSQL local a RDS**

Crea un archivo llamado migrate_to_rds.sh en el computador del usuario con los comandos necesarios para exportar base de datos a SQL, conectarse al RDS creado en AWS y crear la base vacía y finalmente llenarlo. Quedando así la base de datos copiada en la nube. 

In [ ]:
def migrar_datos_postgresql():
    """Script para migrar datos de PostgreSQL local a RDS"""
    print("\n Preparando migración de PostgreSQL local a RDS...")
    
    migration_script = """
#!/bin/bash
# Script de migración PostgreSQL local -> RDS

# Variables
LOCAL_DB="fleetlogix"
LOCAL_USER="postgres"
RDS_ENDPOINT="fleetlogix-db.xxxx.us-east-1.rds.amazonaws.com"
RDS_USER="fleetlogix_admin"
RDS_DB="fleetlogix"

echo " Iniciando migración de base de datos..."

# 1. Hacer dump de la base local
echo " Exportando base de datos local..."
pg_dump -h localhost -U $LOCAL_USER -d $LOCAL_DB -f fleetlogix_dump.sql

# 2. Crear base de datos en RDS
echo " Creando base de datos en RDS..."
psql -h $RDS_ENDPOINT -U $RDS_USER -c "CREATE DATABASE $RDS_DB;"

# 3. Restaurar en RDS
echo " Importando datos en RDS..."
psql -h $RDS_ENDPOINT -U $RDS_USER -d $RDS_DB -f fleetlogix_dump.sql

echo " Migración completada"
"""
    
    with open('migrate_to_rds.sh', 'w') as f:
        f.write(migration_script)
    
    print(" Script de migración creado: migrate_to_rds.sh")
    print("   Ejecutar con: bash migrate_to_rds.sh")

### **2.6. Creación de rol IAM para funciones Lambda**

Le da permisos a: 
1. AWSLambdaBasicExecutionRole para que la Lambda pueda escribir logs en CloudWatch. 
2. AmazonDynamoDBFullAccess le da acceso total a todo DynamoDB.
3. AmazonS3FullAccess le da acceso total a todo S3.
4. AmazonSNSFullAccess le da acceso total a todo SNS.

En caso de ser un caso de empresa real, se deben proteger los datos y no dar acceso completo a todo.

In [ ]:
def crear_rol_iam_lambda():
    """Crear rol IAM para funciones Lambda"""
    print("\n Creando rol IAM para Lambda...")
    
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [{
            "Effect": "Allow",
            "Principal": {"Service": "lambda.amazonaws.com"},
            "Action": "sts:AssumeRole"
        }]
    }
    
    try:
        # Crear rol
        role_response = iam.create_role(
            RoleName='FleetLogixLambdaRole',
            AssumeRolePolicyDocument=json.dumps(trust_policy),
            Description='Rol para funciones Lambda de FleetLogix'
        )
        
        # Adjuntar políticas
        policies = [
            'arn:aws:iam::aws:policy/service-role/AWSLambdaBasicExecutionRole',
            'arn:aws:iam::aws:policy/AmazonDynamoDBFullAccess',
            'arn:aws:iam::aws:policy/AmazonS3FullAccess',
            'arn:aws:iam::aws:policy/AmazonSNSFullAccess'
        ]
        
        for policy in policies:
            iam.attach_role_policy(
                RoleName='FleetLogixLambdaRole',
                PolicyArn=policy
            )
        
        print(" Rol IAM creado: FleetLogixLambdaRole")
        return role_response['Role']['Arn']
        
    except iam.exceptions.EntityAlreadyExistsException:
        print(" Rol IAM ya existe")
        return f"arn:aws:iam::{boto3.client('sts').get_caller_identity()['Account']}:role/FleetLogixLambdaRole"
    except Exception as e:
        print(f" Error creando rol: {e}")
        return None

### **2.7. Ejecución Completa**

1. Crea: RDS + S3 + 4 tablas DynamoDB
2. Configura: Backups + genera el script de migración migrate_to_rds.sh
3. Crea: El rol IAM FleetLogixLambdaRole para las Lambdas
4. Guarda: Todo en un aws_config.json para que las otras partes del código sepan dónde conectarse.

In [ ]:


def main():
    """Ejecutar configuración completa"""
    print("FLEETLOGIX - Configuración AWS")
    print("="*50)
    
    # 1. Crear servicios
    crear_rds_postgresql()
    crear_s3_bucket()
    crear_tablas_dynamodb()
    
    # 2. Configurar
    configurar_backups_automaticos()
    migrar_datos_postgresql()
    
    # 3. Crear rol para Lambda
    rol_arn = crear_rol_iam_lambda()
    
    print("\nCONFIGURACIÓN BÁSICA COMPLETADA")
    print("\nPróximos pasos:")
    print("1. Esperar ~10 min para que RDS esté disponible")
    print("2. Ejecutar script de migración: bash migrate_to_rds.sh")
    print("3. Desplegar funciones Lambda con el rol:", rol_arn)
    print("4. Configurar API Gateway")
    print("5. Configurar triggers automáticos")
    
    # Guardar configuración
    config = {
        'rds_instance': RDS_INSTANCE_ID,
        's3_bucket': S3_BUCKET_NAME,
        'dynamodb_tables': [
            'deliveries_status',
            'vehicle_tracking', 
            'routes_waypoints',
            'alerts_history'
        ],
        'lambda_role_arn': rol_arn,
        'region': AWS_REGION,
        'timestamp': datetime.now().isoformat()
    }
    
    with open('aws_config.json', 'w') as f:
        json.dump(config, f, indent=2)
    
    print("\n Configuración guardada en: aws_config.json")

if __name__ == "__main__":
    main()